<a href="https://colab.research.google.com/github/Talalmh19/NLP-Project/blob/main/NLP_company_policy_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [113]:
 !pip install -q transformers pandas sentence-transformers faiss-cpu

In [123]:
import pandas as pd
import faiss
import re
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

In [115]:
def normalize_arabic(text):
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[\u064B-\u0652]", "", text)
    return text


In [121]:
data_en = [
    "Working Hours Policy: Official working hours are from 9:00 AM to 5:00 PM, Sunday to Thursday.",
    "Annual Leave Policy: Employees are entitled to 30 days of paid annual leave per year.",
    "Remote Work Policy: Remote work is allowed up to two days a week with manager approval.",
    "Sick Leave Policy: Sick leave requires a valid medical certificate if it exceeds two days."
]

data_ar = [
    "سياسة ساعات العمل: ساعات العمل الرسمية من 9 صباحاً حتى 5 مساءً، من الأحد إلى الخميس.",
    "سياسة الإجازات السنوية: يحق للموظفين الحصول على 30 يوماً من الإجازة السنوية المدفوعة سنوياً.",
    "سياسة العمل عن بعد: يسمح بالعمل عن بعد حتى يومين في الأسبوع بموافقة المدير.",
    "سياسة الإجازات المرضية: تتطلب الإجازة المرضية شهادة طبية صالحة إذا تجاوزت يومين."
]

df_en = pd.DataFrame({"policy_text": data_en})
df_ar = pd.DataFrame({
    "policy_text": data_ar,
    "clean_text": [normalize_arabic(t) for t in data_ar]
})

In [124]:
print("Loading models...")
bi_encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

Loading models...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [125]:
emb_en = bi_encoder.encode(df_en["policy_text"].tolist(), convert_to_numpy=True).astype(np.float32)
faiss.normalize_L2(emb_en)
index_en = faiss.IndexFlatIP(emb_en.shape[1])
index_en.add(emb_en)

emb_ar = bi_encoder.encode(df_ar["clean_text"].tolist(), convert_to_numpy=True).astype(np.float32)
faiss.normalize_L2(emb_ar)
index_ar = faiss.IndexFlatIP(emb_ar.shape[1])
index_ar.add(emb_ar)

In [126]:
def ask_policy_bot(query):
    is_arabic = bool(re.search(r'[\u0600-\u06FF]', query))

    if is_arabic:
        clean_q = normalize_arabic(query)
        q_emb = bi_encoder.encode([clean_q], convert_to_numpy=True).astype(np.float32)
        faiss.normalize_L2(q_emb)

        distances, indices = index_ar.search(q_emb, 3)
        candidates = [df_ar["policy_text"].iloc[idx] for idx in indices[0]]

        pairs = [[query, cand] for cand in candidates]
        scores = reranker.predict(pairs)
        best_match = candidates[np.argmax(scores)]

    else:
        q_emb = bi_encoder.encode([query], convert_to_numpy=True).astype(np.float32)
        faiss.normalize_L2(q_emb)

        distances, indices = index_en.search(q_emb, 3)
        candidates = [df_en["policy_text"].iloc[idx] for idx in indices[0]]

        pairs = [[query, cand] for cand in candidates]
        scores = reranker.predict(pairs)
        best_match = candidates[np.argmax(scores)]

    return best_match

In [127]:
print("\n--- Chatbot Testing ---")
queries = [
    "كم يوم إجازة عندي؟",               # How many days of leave do I have?
    "Can I work from home?",            # Can I work from home?
    "متى يبدأ الدوام؟"                  # When do working hours start?
]

for q in queries:
    print(f"Employee asks: {q}")
    print(f"Bot answers: {ask_policy_bot(q)}")
    print("-" * 40)


--- Chatbot Testing ---
Employee asks: كم يوم إجازة عندي؟
Bot answers: سياسة الإجازات السنوية: يحق للموظفين الحصول على 30 يوماً من الإجازة السنوية المدفوعة سنوياً.
----------------------------------------
Employee asks: Can I work from home?
Bot answers: Remote Work Policy: Remote work is allowed up to two days a week with manager approval.
----------------------------------------
Employee asks: متى يبدأ الدوام؟
Bot answers: سياسة ساعات العمل: ساعات العمل الرسمية من 9 صباحاً حتى 5 مساءً، من الأحد إلى الخميس.
----------------------------------------
